In [2]:
import pandas as pd
from torch.utils.data import Subset
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import ast
pd.set_option("display.max_columns", 500)

In [3]:
df = pd.read_csv("../reports/padchest/layer/predictions_per_layer_test.csv")
df.describe()

,y_true,layer_1,layer_2,layer_3,layer_4,layer_5,layer_6,layer_7,layer_8,layer_9,layer_10,layer_11,layer_12,layer_13,layer_14,layer_15,layer_16,layer_17
count,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000,21770.000000
mean,0.085714,0.525523,0.564067,0.494321,0.614346,0.654545,0.605022,0.594061,0.417487,0.393533,0.413643,0.440069,0.405062,0.430407,0.423410,0.369938,0.348844,0.325976
std,0.279948,0.070424,0.064727,0.067594,0.110505,0.096575,0.098639,0.092234,0.146543,0.141625,0.143504,0.155539,0.146190,0.138855,0.249307,0.244526,0.229698,0.228254
min,0.000000,0.231967,0.267870,0.205620,0.217119,0.277030,0.240223,0.246360,0.052320,0.061044,0.064706,0.064724,0.060430,0.075716,0.009776,0.011948,0.028405,0.026153
25%,0.000000,0.481432,0.525522,0.453178,0.545087,0.596094,0.541215,0.534327,0.306542,0.284630,0.305375,0.323226,0.294099,0.326806,0.210010,0.167522,0.169601,0.150595
50%,0.000000,0.534003,0.571936,0.501502,0.629410,0.668393,0.615383,0.604297,0.421084,0.392787,0.415174,0.444456,0.403487,0.430289,0.394265,0.314099,0.282712,0.256240
75%,0.000000,0.576823,0.611374,0.543202,0.697846,0.726471,0.678643,0.663834,0.527481,0.499634,0.522803,0.559331,0.514510,0.533040,0.611812,0.530051,0.473958,0.441851
max,1.000000,0.729174,0.769699,0.766134,0.966972,0.967349,0.946151,0.905071,0.886455,0.853888,0.845936,0.866867,0.846814,0.838381,0.997624,0.999679,0.999004,0.999151


# PADCHEST

In [4]:
train_df = pd.read_csv("../padchest_splits/padchest/train_split.csv")
test_df = pd.read_csv("../padchest_splits/padchest/test_split.csv")
val_df = pd.read_csv("../padchest_splits/padchest/val_split.csv")

In [5]:
print(len(train_df))
print(len(val_df))
print(len(test_df))
print("all in all:")
print(len(train_df) + len(val_df) + len(test_df))

69122
17194
21770
all in all:
108086


## cardiomegaly numbers

In [6]:
cardio_counts_train = pd.crosstab(train_df["PatientSex_DICOM"], train_df["target_label"])
cardio_counts_train = cardio_counts_train.rename(columns={0: "non_cardiomegaly", 1: "cardiomegaly"})


cardio_ratios_train = cardio_counts_train.div(cardio_counts_train.sum(axis=1), axis=0)
cardio_ratios_train = cardio_ratios_train.add_suffix("_ratio")


cardio_summary_train = pd.concat([cardio_counts_train, cardio_ratios_train], axis=1)
cardio_summary_train

target_label,non_cardiomegaly,cardiomegaly,non_cardiomegaly_ratio,cardiomegaly_ratio
PatientSex_DICOM,,,,
F,30862,3518,0.897673,0.102327
M,32030,2705,0.922125,0.077875
O,6,1,0.857143,0.142857


In [7]:
cardio_counts_test = pd.crosstab(test_df["PatientSex_DICOM"], test_df["target_label"])
cardio_counts_test = cardio_counts_test.rename(columns={0: "non_cardiomegaly", 1: "cardiomegaly"})


cardio_ratios_test = cardio_counts_test.div(cardio_counts_test.sum(axis=1), axis=0)
cardio_ratios_test = cardio_ratios_test.add_suffix("_ratio")


cardio_summary_test = pd.concat([cardio_counts_test, cardio_ratios_test], axis=1)
cardio_summary_test

target_label,non_cardiomegaly,cardiomegaly,non_cardiomegaly_ratio,cardiomegaly_ratio
PatientSex_DICOM,,,,
F,9716,1044,0.902974,0.097026
M,10183,822,0.925307,0.074693
O,3,0,1.000000,0.000000


In [8]:
cardio_counts_val = pd.crosstab(val_df["PatientSex_DICOM"], val_df["target_label"])
cardio_counts_val = cardio_counts_val.rename(columns={0: "non_cardiomegaly", 1: "cardiomegaly"})


cardio_ratios_val = cardio_counts_val.div(cardio_counts_val.sum(axis=1), axis=0)
cardio_ratios_val = cardio_ratios_val.add_suffix("_ratio")


cardio_summary_val = pd.concat([cardio_counts_val, cardio_ratios_val], axis=1)
cardio_summary_val

target_label,non_cardiomegaly,cardiomegaly,non_cardiomegaly_ratio,cardiomegaly_ratio
PatientSex_DICOM,,,,
F,7620,881,0.896365,0.103635
M,7999,689,0.920695,0.079305
O,4,0,1.000000,0.000000


## pneumothorax numbers

In [12]:
test_df_px = pd.read_csv("../padchest_splits/padchest_px/test_split.csv")
train_df_px = pd.read_csv("../padchest_splits/padchest_px/train_split.csv")
val_df_px = pd.read_csv("../padchest_splits/padchest_px/val_split.csv")

In [13]:
counts_train_px = pd.crosstab(train_df_px["PatientSex_DICOM"], train_df_px["target_label"])
counts_train_px = counts_train_px.rename(columns={0: "non_pneumothorax", 1: "pneumothorax"})


ratios_train_px = counts_train_px.div(counts_train_px.sum(axis=1), axis=0)
ratios_train_px = ratios_train_px.add_suffix("_ratio")


summary_train_px = pd.concat([counts_train_px, ratios_train_px], axis=1)
summary_train_px

target_label,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
PatientSex_DICOM,,,,
F,34287,93,0.997295,0.002705
M,34580,155,0.995538,0.004462
O,7,0,1.000000,0.000000


In [14]:
counts_test_px = pd.crosstab(test_df_px["PatientSex_DICOM"], test_df_px["target_label"])
counts_test_px = counts_test_px.rename(columns={0: "non_pneumothorax", 1: "pneumothorax"})


ratios_test_px = counts_test_px.div(counts_test_px.sum(axis=1), axis=0)
ratios_test_px = ratios_test_px.add_suffix("_ratio")


summary_test_px = pd.concat([counts_test_px, ratios_test_px], axis=1)
summary_test_px

target_label,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
PatientSex_DICOM,,,,
F,10734,26,0.997584,0.002416
M,10949,56,0.994911,0.005089
O,3,0,1.000000,0.000000


In [15]:
counts_val_px = pd.crosstab(val_df_px["PatientSex_DICOM"], val_df_px["target_label"])
counts_val_px = counts_val_px.rename(columns={0: "non_pneumothorax", 1: "pneumothorax"})


ratios_val_px = counts_val_px.div(counts_val_px.sum(axis=1), axis=0)
ratios_val_px = ratios_val_px.add_suffix("_ratio")


summary_val_px = pd.concat([counts_val_px, ratios_val_px], axis=1)
summary_val_px

target_label,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
PatientSex_DICOM,,,,
F,8483,18,0.997883,0.002117
M,8653,35,0.995971,0.004029
O,4,0,1.000000,0.000000


# Manufacturer_DICOM

In [16]:
counts_train_px = pd.crosstab(train_df_px["Manufacturer_DICOM"], train_df_px["target_label"])
counts_train_px = counts_train_px.rename(columns={0: "non_pneumothorax", 1: "pneumothorax"})


ratios_train_px = counts_train_px.div(counts_train_px.sum(axis=1), axis=0)
ratios_train_px = ratios_train_px.add_suffix("_ratio")


summary_train_px = pd.concat([counts_train_px, ratios_train_px], axis=1)
summary_train_px

target_label,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
Manufacturer_DICOM,,,,
ImagingDynamicsCompanyLtd,34654,29,0.999164,0.000836
PhilipsMedicalSystems,34220,219,0.993641,0.006359


In [17]:
counts_test_px = pd.crosstab(test_df_px["Manufacturer_DICOM"], test_df_px["target_label"])
counts_test_px = counts_test_px.rename(columns={0: "non_pneumothorax", 1: "pneumothorax"})


ratios_test_px = counts_test_px.div(counts_test_px.sum(axis=1), axis=0)
ratios_test_px = ratios_test_px.add_suffix("_ratio")


summary_test_px = pd.concat([counts_test_px, ratios_test_px], axis=1)
summary_test_px

target_label,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
Manufacturer_DICOM,,,,
ImagingDynamicsCompanyLtd,10966,4,0.999635,0.000365
PhilipsMedicalSystems,10722,78,0.992778,0.007222


In [18]:
counts_val_px = pd.crosstab(val_df_px["Manufacturer_DICOM"], val_df_px["target_label"])
counts_val_px = counts_val_px.rename(columns={0: "non_pneumothorax", 1: "pneumothorax"})


ratios_val_px = counts_val_px.div(counts_val_px.sum(axis=1), axis=0)
ratios_val_px = ratios_val_px.add_suffix("_ratio")


summary_val_px = pd.concat([counts_val_px, ratios_val_px], axis=1)
summary_val_px

target_label,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
Manufacturer_DICOM,,,,
ImagingDynamicsCompanyLtd,8685,6,0.999310,0.000690
PhilipsMedicalSystems,8456,47,0.994473,0.005527


# ChestX-ray14

In [19]:
chestx_train_and_val = pd.read_csv("../data/processed/ChestX-ray14/train/files/processed_labels_train.csv")
test_chestx = pd.read_csv("../data/processed/ChestX-ray14/test/files/processed_labels_drains.csv")



def get_train_val_split(df, val_split=0.2, random_state=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=val_split, random_state=random_state)
    train_idx, val_idx = next(
        gss.split(X=df["Image Index"], y=df["Pneumothorax"], groups=df["Patient ID"])
    )

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    return train_df, val_df

train_chestx, val_chestx = get_train_val_split(chestx_train_and_val)



In [20]:
print(len(train_chestx))
print(len(val_chestx))
print(len(test_chestx))
print("all in all:")
print(len(train_chestx) + len(val_chestx) + len(test_chestx))

69625
16899
25596
all in all:
112120


In [21]:
counts_train_chestx = pd.crosstab(train_chestx["Patient Gender"], train_chestx["Pneumothorax"])
counts_train_chestx = counts_train_chestx.rename(columns={0: "non_pneumothorax",1: "pneumothorax"})


ratios_train_chestx = counts_train_chestx.div(counts_train_chestx.sum(axis=1), axis=0)
ratios_train_chestx = ratios_train_chestx.add_suffix("_ratio")


summary_train_chestx = pd.concat([counts_train_chestx, ratios_train_chestx], axis=1)
summary_train_chestx

Pneumothorax,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
Patient Gender,,,,
F,29148,1111,0.963284,0.036716
M,38314,1052,0.973276,0.026724


In [22]:
counts_val_chestx = pd.crosstab(val_chestx["Patient Gender"], val_chestx["Pneumothorax"])
counts_val_chestx = counts_val_chestx.rename(columns={0: "non_pneumothorax",1: "pneumothorax"})


ratios_val_chestx = counts_val_chestx.div(counts_val_chestx.sum(axis=1), axis=0)
ratios_val_chestx = ratios_val_chestx.add_suffix("_ratio")


summary_val_chestx = pd.concat([counts_val_chestx, ratios_val_chestx], axis=1)
summary_val_chestx

Pneumothorax,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
Patient Gender,,,,
F,7564,243,0.968874,0.031126
M,8861,231,0.974593,0.025407


In [23]:
counts_test_chestx = pd.crosstab(test_chestx["Patient Gender"], test_chestx["Pneumothorax"])
counts_test_chestx = counts_test_chestx.rename(columns={0: "non_pneumothorax",1: "pneumothorax"})


ratios_test_chestx = counts_test_chestx.div(counts_test_chestx.sum(axis=1), axis=0)
ratios_test_chestx = ratios_test_chestx.add_suffix("_ratio")


summary_test_chestx = pd.concat([counts_test_chestx, ratios_test_chestx], axis=1)
summary_test_chestx

Pneumothorax,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
Patient Gender,,,,
F,9483,1231,0.885104,0.114896
M,13448,1434,0.903642,0.096358


# drains

In [27]:
counts_test_chestx = pd.crosstab(test_chestx["Drain"], test_chestx["Pneumothorax"])
counts_test_chestx = counts_test_chestx.rename(columns={0: "non_pneumothorax",1: "pneumothorax"})


ratios_test_chestx = counts_test_chestx.div(counts_test_chestx.sum(axis=1), axis=0)
ratios_test_chestx = ratios_test_chestx.add_suffix("_ratio")


summary_test_chestx = pd.concat([counts_test_chestx, ratios_test_chestx], axis=1)
summary_test_chestx

Pneumothorax,non_pneumothorax,pneumothorax,non_pneumothorax_ratio,pneumothorax_ratio
Drain,,,,
0.0,12437,973,0.927442,0.072558
1.0,10494,1692,0.861152,0.138848
